# Audit DB Intermediate: How to Design an Auditable Schema

## Modeling Runs, Spans, and Tool Calls

**Difficulty: Intermediate | ~40 min | Requires Lab 1 (Recording Agent Activity) done in this Supabase project**

*Lab 2 of 8 in the Audit DB Labs module.*

Lab 1 gave the harness a single flat `event` table — good enough to write history reliably, but it can't tell you *what kind* of thing happened without parsing a free-text `event_type` string, and nothing stops a bad row from pointing at a run that doesn't exist. This lab replaces that with the real shape the README's Section 2 hierarchy describes: `run` → `span` → `tool_call` / `guardrail_event`.

You will, step by step, on the same Supabase Postgres and the same `support-agent` story:

1. Confirm Lab 1's `run` table is already there — this lab builds on top of it, not instead of it.
2. Create three new tables — `span`, `tool_call`, `guardrail_event` — wired together with real `FOREIGN KEY`, `NOT NULL`, and `CHECK` constraints, plus indexes for the lookups you'll actually run.
3. Record a run with two spans, one tool call, and two guardrail events (one passing, one failing).
4. Watch the `CHECK` constraint reject an out-of-range value.
5. Reconstruct one run's full nested history — run → spans → each span's tool calls and guardrail events — using the indexes you just built.

Everything runs against your real database, so every id and timestamp you see once you run this yourself is genuine, not made up.

In [ ]:
!pip install python-dotenv==1.2.3 psycopg2-binary==2.9.12

One pinned line installs everything the lab needs, matching Section 9 exactly.

### Step 1 — Connect to Postgres

In [11]:
import os
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
import psycopg2

# Walk upward until we find the module's .env so credentials stay out of this notebook.
env_path = next((p / ".env" for p in [Path.cwd(), *Path.cwd().parents] if (p / ".env").exists()), None)
load_dotenv(env_path)
database_url = os.getenv("DATABASE_URL")
assert database_url, "DATABASE_URL not found - complete README Section 7 setup first."

connection = psycopg2.connect(database_url, connect_timeout=10)
cursor = connection.cursor()
cursor.execute("SELECT version();")
print(cursor.fetchone()[0])

PostgreSQL 17.6 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 15.2.0, 64-bit


Same connection pattern as Lab 1: walk upward for the module-level `.env`, load `DATABASE_URL`, connect with `psycopg2`, and print the version string as proof the connection works before anything else runs.

### Step 2 — Confirm Lab 1's Schema Is Already Here

In [12]:
# Fail-fast: Lab 2 builds on Lab 1's run table
cursor.execute("SELECT to_regclass('public.run')")
assert cursor.fetchone()[0] is not None, "run table missing - complete Lab 1 in this Supabase project first."
print("Lab 1's run table found - building the schema on top of it.")

Lab 1's run table found - building the schema on top of it.


This lab's Prerequisites assume Lab 1 was completed in this same Supabase project, so the `run` table should already exist. `to_regclass` asks Postgres whether a table exists without raising an error if it doesn't — a cleaner check than trying a query and catching the exception. Failing fast here with a clear message beats a confusing "relation does not exist" error three cells from now.

### Step 3 — Model the Hierarchy with Real Constraints

In [13]:
# DDL: three tables with FOREIGN KEY, NOT NULL, CHECK, and indexes
create_schema = """
    CREATE TABLE IF NOT EXISTS span (
        span_id     SERIAL PRIMARY KEY,
        run_id      INTEGER NOT NULL REFERENCES run(run_id),
        span_name   TEXT NOT NULL,
        started_at  TIMESTAMPTZ NOT NULL DEFAULT now(),
        ended_at    TIMESTAMPTZ);
    CREATE TABLE IF NOT EXISTS tool_call (
        tool_call_id SERIAL PRIMARY KEY,
        span_id      INTEGER NOT NULL REFERENCES span(span_id),
        tool_name    TEXT NOT NULL,
        arguments    TEXT,
        result       TEXT,
        success      BOOLEAN NOT NULL,
        called_at    TIMESTAMPTZ NOT NULL DEFAULT now());
    CREATE TABLE IF NOT EXISTS guardrail_event (
        guardrail_event_id SERIAL PRIMARY KEY,
        span_id             INTEGER NOT NULL REFERENCES span(span_id),
        check_name          TEXT NOT NULL,
        outcome             TEXT NOT NULL CHECK (outcome IN ('pass', 'fail', 'warn')),
        reason              TEXT,
        checked_at          TIMESTAMPTZ NOT NULL DEFAULT now());
    CREATE INDEX IF NOT EXISTS idx_span_run_id ON span (run_id);
    CREATE INDEX IF NOT EXISTS idx_tool_call_span_id ON tool_call (span_id);
    CREATE INDEX IF NOT EXISTS idx_guardrail_event_span_id ON guardrail_event (span_id);
"""
cursor.execute(create_schema)
connection.commit()
print("Tables ready: span, tool_call, guardrail_event (with foreign keys, CHECK, and indexes).")

Tables ready: span, tool_call, guardrail_event (with foreign keys, CHECK, and indexes).


Three new tables, each pointing at its parent through a `FOREIGN KEY` instead of a plain integer column: `span.run_id` references `run(run_id)`, and both `tool_call.span_id` and `guardrail_event.span_id` reference `span(span_id)`. Postgres will now refuse to insert a child row whose parent doesn't exist — Lab 1's `event` table couldn't make that promise.

`NOT NULL` marks the fields a row can't be meaningful without (a tool call with no `tool_name`, or a guardrail event with no `outcome`, isn't useful data). The `CHECK` constraint on `guardrail_event.outcome` is the schema enforcing its own vocabulary: only `'pass'`, `'fail'`, or `'warn'` are legal values, so a typo or a bad enum value is rejected at write time instead of silently corrupting downstream reports.

The three `CREATE INDEX` statements target the exact lookups this lab and the next one actually run: "all spans for a run" (`span.run_id`) and "all tool_calls / guardrail_events for a span" — the same pattern Step 9 uses to reconstruct history. Without them, those lookups force Postgres to scan the whole table as it grows.

### Step 4 — Start a New Run

In [14]:
# Create the run header first — all spans reference this run_id
cursor.execute(
    "INSERT INTO run (agent_name, status) VALUES (%s, %s) RETURNING run_id",
    ("support-agent", "running"),
)
current_run_id = cursor.fetchone()[0]
connection.commit()
print(f"Run {current_run_id} started for support-agent.")

Run 38 started for support-agent.


Same `run` table Lab 1 built, same insert pattern: create the header row first so `RETURNING run_id` hands back the id every span below needs.

### Step 5 — Log a Retrieval Span and Its Tool Call

In [15]:
# Span 1: retrieval step — one tool call nested inside it
cursor.execute(
    "INSERT INTO span (run_id, span_name) VALUES (%s, %s) RETURNING span_id",
    (current_run_id, "retrieve_orders"),
)
retrieve_span_id = cursor.fetchone()[0]

cursor.execute(
    "INSERT INTO tool_call (span_id, tool_name, arguments, result, success) "
    "VALUES (%s, %s, %s, %s, %s)",
    (retrieve_span_id, "query_orders_db",
     '{"table": "orders", "filter": "shipped_yesterday"}', "1284 rows returned", True),
)

# Close the span the same way Lab 1 closed a run: one lifecycle update when the work is done.
cursor.execute("UPDATE span SET ended_at = now() WHERE span_id = %s", (retrieve_span_id,))
connection.commit()
print(f"Span {retrieve_span_id} 'retrieve_orders' recorded with 1 tool_call, then closed.")

Span 9 'retrieve_orders' recorded with 1 tool_call, then closed.


A span is a unit of work with its own lifecycle: insert it, do the work it represents, then close it — the same one-time lifecycle update Lab 1 used to close a run. Here the span represents one retrieval step, and the `tool_call` row nested inside it records the actual database query the agent ran: what tool, what arguments, what came back, whether it succeeded.

### Step 6 — Log an Answer Span with Two Guardrail Checks

In [16]:
# Span 2: answer step — two guardrail checks (pass + fail)
cursor.execute(
    "INSERT INTO span (run_id, span_name) VALUES (%s, %s) RETURNING span_id",
    (current_run_id, "generate_answer"),
)
answer_span_id = cursor.fetchone()[0]

cursor.execute(
    "INSERT INTO guardrail_event (span_id, check_name, outcome, reason) VALUES (%s, %s, %s, %s)",
    (answer_span_id, "pii_scan", "pass", "no personal data found in the draft answer"),
)
cursor.execute(
    "INSERT INTO guardrail_event (span_id, check_name, outcome, reason) VALUES (%s, %s, %s, %s)",
    (answer_span_id, "prompt_injection_scan", "fail",
     "retrieved content contained an instruction-override attempt"),
)

cursor.execute("UPDATE span SET ended_at = now() WHERE span_id = %s", (answer_span_id,))
connection.commit()
print(f"Span {answer_span_id} 'generate_answer' recorded with 2 guardrail_events (pass, fail), then closed.")

Span 10 'generate_answer' recorded with 2 guardrail_events (pass, fail), then closed.


This span represents the model producing its answer. Two policy checks fired during it: a PII scan that passed, and a prompt-injection scan that failed because the retrieved content tried to smuggle in an instruction. Logging both — not just the failure — is what lets an auditor later see every check that ran, not only the ones that blocked something.

### Step 7 — Prove the CHECK Constraint Fires

In [17]:
# Prove the CHECK constraint rejects invalid outcome values
try:
    cursor.execute(
        "INSERT INTO guardrail_event (span_id, check_name, outcome, reason) VALUES (%s, %s, %s, %s)",
        (answer_span_id, "bad_test", "maybe", "attempting an out-of-range outcome value"),
    )
except psycopg2.errors.CheckViolation as error:
    connection.rollback()
    print("CHECK constraint rejected the out-of-range outcome, as designed:")
    print(error)

CHECK constraint rejected the out-of-range outcome, as designed:
new row for relation "guardrail_event" violates check constraint "guardrail_event_outcome_check"
DETAIL:  Failing row contains (9, 10, bad_test, maybe, attempting an out-of-range outcome value, 2026-08-25 04:27:44.835453+00).



`'maybe'` isn't one of the three allowed outcomes, so Postgres should refuse the insert rather than silently accepting a value nothing downstream knows how to interpret. A failed statement leaves the whole transaction in an aborted state until `rollback()` clears it — that's why the `except` block calls it before the notebook does anything else with this connection. `print(error)` shows the full, unsanitized message Postgres returned — including its `DETAIL:` line naming the actual failing row — because seeing the real database error is the point of a learning lab, not a summarized one-liner.

### Step 8 — Close the Run

In [18]:
cursor.execute(
    "UPDATE run SET ended_at = now(), status = 'completed', total_cost = %s WHERE run_id = %s",
    (0.0057, current_run_id),
)
connection.commit()
print(f"Run {current_run_id} closed: status='completed', total_cost=$0.0057")

Run 38 closed: status='completed', total_cost=$0.0057


Same lifecycle update Lab 1 used: this is the only write the `run` header ever gets after creation, now that both of its spans are closed.

### Step 9 — Reconstruct the Run's Full Nested History

In [19]:
# Walk the hierarchy: run -> spans -> tool_calls + guardrail_events per span
cursor.execute(
    "SELECT span_id, span_name, started_at, ended_at FROM span WHERE run_id = %s ORDER BY started_at",
    (current_run_id,),
)
spans = cursor.fetchall()
print(f"Run {current_run_id} has {len(spans)} span(s):")
for span_id, span_name, started_at, ended_at in spans:
    print(f"  span {span_id} - {span_name} ({started_at} -> {ended_at})")

    cursor.execute(
        "SELECT tool_call_id, tool_name, result, success FROM tool_call WHERE span_id = %s",
        (span_id,),
    )
    for tool_call_id, tool_name, result, success in cursor.fetchall():
        print(f"    tool_call {tool_call_id}: {tool_name} -> {result} (success={success})")

    cursor.execute(
        "SELECT guardrail_event_id, check_name, outcome, reason FROM guardrail_event WHERE span_id = %s",
        (span_id,),
    )
    for guardrail_event_id, check_name, outcome, reason in cursor.fetchall():
        print(f"    guardrail_event {guardrail_event_id}: {check_name} -> {outcome} ({reason})")

Run 38 has 2 span(s):
  span 9 - retrieve_orders (2026-08-25 04:27:42.385864+00:00 -> 2026-08-25 04:27:42.385864+00:00)
    tool_call 6: query_orders_db -> 1284 rows returned (success=True)
  span 10 - generate_answer (2026-08-25 04:27:43.499745+00:00 -> 2026-08-25 04:27:43.499745+00:00)
    guardrail_event 7: pii_scan -> pass (no personal data found in the draft answer)
    guardrail_event 8: prompt_injection_scan -> fail (retrieved content contained an instruction-override attempt)


This is the query the whole schema exists to answer: for this run, what spans ran, and inside each span, what tool calls and guardrail events happened? Each lookup below hits the index built in Step 3 instead of scanning the full table — `span.run_id`, then `tool_call.span_id` and `guardrail_event.span_id` per span. A single `JOIN` could do this in one round trip (Lab 3's territory); walking the foreign keys directly here makes the parent → child relationship itself the thing you're seeing work.

### Step 10 — Recap

In [20]:
cursor.close()
connection.close()
print(f"Run {current_run_id}: 1 run, 2 spans, 1 tool_call, 2 guardrail_events - schema is now relational.")
print("Foreign keys, NOT NULL, CHECK, and indexes make the shape self-protecting, not just documented.")

Run 38: 1 run, 2 spans, 1 tool_call, 2 guardrail_events - schema is now relational.
Foreign keys, NOT NULL, CHECK, and indexes make the shape self-protecting, not just documented.


Close the connection, then state what this run's schema now proves: the same story Lab 1 told, but shaped so the database itself — not just careful code — refuses to let the pieces come apart.